In [1]:
!pip install lightgbm

In [17]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,classification_report


In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e11/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e11/test.csv')

print(train.shape)
print(test.shape)

(699960, 26)
(299984, 25)


methanadi nikamma text tika int karala vitharrai thiyenne.ara loku agaya parasa 0-1 atharata ganna vade devanin vate try kaannna

In [4]:
from sklearn.preprocessing import OrdinalEncoder


cat_cols = train.select_dtypes(include=['object']).columns.tolist()

# Target column එක string එකක් නම් ඒක මේ list එකෙන් අයින් කරන්න (උදා: 'Class')
if 'Class' in cat_cols:
    cat_cols.remove('Class')

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

train[cat_cols] = encoder.fit_transform(train[cat_cols].astype(str))
test[cat_cols] = encoder.transform(test[cat_cols].astype(str))

In [5]:
train.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,3.0,4.0,7.0,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,4.0,2.0,1.0,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,3.0,3.0,5.0,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,0.0,4.0,1.0,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,0.0,3.0,6.0,False,False,False,False,False,False,0.0


In [6]:
test.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt
0,699960,28.21,4.52,4,17.87,37,45,1483.49,0,15.4,...,4.0,3.0,3.0,7.0,False,False,False,False,False,False
1,699961,644.90,5.48,1,6.68,52,28,2183.64,0,11.2,...,3.0,4.0,1.0,1.0,False,False,False,False,False,False
2,699962,20.53,12.20,3,15.39,44,75,1208.59,0,20.8,...,4.0,2.0,3.0,7.0,False,False,False,False,False,False
3,699963,10.83,6.90,2,44.34,50,54,1790.85,0,9.2,...,2.0,4.0,1.0,2.0,False,False,False,False,False,False
4,699964,19.00,0.63,3,25.22,50,66,1676.18,0,20.0,...,2.0,4.0,2.0,1.0,False,False,False,False,False,False


In [7]:
x = train.drop(columns=["is_fraud"])
y = train["is_fraud"]

In [8]:
x.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,2.0,3.0,4.0,7.0,False,False,False,False,False,False
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,4.0,4.0,2.0,1.0,False,False,False,False,False,False
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,4.0,3.0,3.0,5.0,False,False,True,False,False,False
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,2.0,0.0,4.0,1.0,False,False,False,False,False,False
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,4.0,0.0,3.0,6.0,False,False,False,False,False,False


In [9]:
y.head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: is_fraud, dtype: float64

In [10]:
print(train.select_dtypes(include=['object']).columns)

Index([], dtype='object')


In [12]:
params = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=40,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    reg_alpha=0.1,
    reg_lambda=0.2,
    n_estimators=5000,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model = lgb.LGBMClassifier(
    n_estimators=5000,
    random_state = 42,
    max_depth = 12,
    n_jobs = -1
)

model.fit(x,y)
print("model is traind!")

[LightGBM] [Info] Number of positive: 11689, number of negative: 688271
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.047864 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1995
[LightGBM] [Info] Number of data points in the train set: 699960, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.016700 -> initscore=-4.075534
[LightGBM] [Info] Start training from score -4.075534
model is traind!


In [13]:
pred = model.predict(test)
print(pred)

[0. 0. 0. ... 0. 0. 0.]


In [21]:
submission.to_csv("submission.csv",index=False)

In [20]:
submission = pd.DataFrame({"id": test['id'] , "is_fraud": pred})
print(submission)

            id  is_fraud
0       699960       0.0
1       699961       0.0
2       699962       0.0
3       699963       0.0
4       699964       0.0
...        ...       ...
299979  999939       0.0
299980  999940       0.0
299981  999941       0.0
299982  999942       0.0
299983  999943       0.0

[299984 rows x 2 columns]
